# Overnight Water Shortage Prediction – MLOps\nRun the cells from top to bottom. This notebook creates synthetic datasets, engineers features, trains and evaluates models, logs the selected model with MLflow, runs validation checks, and starts a FastAPI service.

In [ ]:
!pip install -q mlflow fastapi uvicorn scikit-learn pandas numpy joblib

In [ ]:
import os, numpy as np, pandas as pd\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.tree import DecisionTreeClassifier\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix\nnp.random.seed(42)\nNUM_RECORDS, TANK_CAPACITY = 250, 100000

In [ ]:
def generate_dataset(version='v1'):\n    tank = np.random.uniform(20000, 100000, NUM_RECORDS)\n    homes = np.random.randint(50, 150, NUM_RECORDS)\n    factor = 1.35 if version == 'v2' else 1.0\n    use = np.random.uniform(15, 30, NUM_RECORDS) * factor\n    pump_hours = np.random.uniform(0, 4, NUM_RECORDS)\n    pump_rate = np.random.uniform(8000, 12000, NUM_RECORDS)\n    leak = np.random.uniform(100, 500, NUM_RECORDS)\n    day = np.random.randint(0, 2, NUM_RECORDS)\n    morning = np.minimum(tank - homes*use*10 + pump_hours*pump_rate - leak*10, TANK_CAPACITY)\n    return pd.DataFrame({'tank_level_8pm':tank,'households':homes,'avg_hourly_consumption':use,'pumping_duration':pump_hours,'pumping_rate':pump_rate,'leakage_rate':leak,'day_type':day,'morning_water_shortage':(morning<15000).astype(int)})\ndf_v1, df_v2 = generate_dataset('v1'), generate_dataset('v2')\nos.makedirs('data', exist_ok=True)\ndf_v1.to_csv('data/water_tank_v1.csv', index=False)\ndf_v2.to_csv('data/water_tank_v2.csv', index=False)\nprint('V1:', df_v1.shape, 'V2:', df_v2.shape)\ndisplay(df_v2.head())

In [ ]:
def preprocess_data(df):\n    out=df.copy()\n    out['expected_overnight_demand']=out['households']*out['avg_hourly_consumption']*10\n    out['expected_pump_supply']=out['pumping_duration']*out['pumping_rate']\n    out['projected_water_buffer']=out['tank_level_8pm']-out['expected_overnight_demand']-out['leakage_rate']*10+out['expected_pump_supply']\n    return out\nengineered_data=preprocess_data(df_v2)\nengineered_data.to_csv('data/feature_store.csv', index=False)\ndisplay(engineered_data.head())

In [ ]:
features=['tank_level_8pm','households','avg_hourly_consumption','pumping_duration','pumping_rate','leakage_rate','day_type','expected_overnight_demand','expected_pump_supply','projected_water_buffer']\nX,y=engineered_data[features],engineered_data['morning_water_shortage']\nX_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)\nmodels={'Decision Tree':DecisionTreeClassifier(max_depth=5,random_state=42),'Random Forest':RandomForestClassifier(n_estimators=50,random_state=42)}\nresults={}\nfor name,model in models.items():\n    model.fit(X_train,y_train); pred=model.predict(X_test)\n    results[name]={'model':model,'Accuracy':accuracy_score(y_test,pred),'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred,zero_division=0),'F1 Score':f1_score(y_test,pred,zero_division=0)}\n    print(name, {k:v for k,v in results[name].items() if k!='model'})\nbest_name=max(results,key=lambda n:results[n]['F1 Score'])\nbest_model=results[best_name]['model']\nprint('Selected:',best_name)

In [ ]:
import mlflow, mlflow.sklearn\nmlflow.set_experiment('Water_Shortage_Prediction')\nwith mlflow.start_run():\n    mlflow.log_param('model_name',best_name)\n    mlflow.log_param('dataset_version','v2')\n    for metric in ['Accuracy','Precision','Recall','F1 Score']:\n        mlflow.log_metric(metric.lower().replace(' ', '_'),results[best_name][metric])\n    mlflow.sklearn.log_model(best_model,name='water_shortage_model',skops_trusted_types=['sklearn.tree._tree.Tree'])\nprint('MLflow tracking completed.')

In [ ]:
assert df_v2['tank_level_8pm'].between(0,TANK_CAPACITY).all()\nassert (df_v2['households']>0).all()\nassert len(best_model.predict(X_test))==len(y_test)\nprint('All validation tests passed.')

## FastAPI serving\nFor a simple Colab demonstration, the API code can be run in a separate cell using FastAPI and Uvicorn. See `app.py` in the repository for the standalone local API implementation.